# Quick-Start 01: Q-Learning on CartPole-v1

**Estimated time: under 2 minutes**

## Learning Objectives

By the end of this notebook you will:
1. Understand why continuous state spaces require discretization for tabular Q-learning
2. Train a Q-table agent on CartPole-v1 and observe reward improvement over 500 episodes
3. Identify where tabular methods break down and motivate the need for function approximation

## Prerequisites

- Quick-Start 00 (the RL loop, Q-learning update rule)
- Basic NumPy

## Dependencies

```
pip install gymnasium numpy matplotlib
```

---

## Setup

We fix all random seeds up front. The gymnasium seed is set at environment creation so
the reset trajectory is reproducible.

In [ ]:
# Purpose: Imports and reproducibility setup
# Key Concept: Two random streams to seed — NumPy (our code) and gymnasium (the env)

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from collections import defaultdict

np.random.seed(42)

print("gymnasium version:", gym.__version__)
print("numpy version:    ", np.__version__)

## The CartPole Environment

CartPole-v1 is the canonical benchmark for simple RL. A pole is balanced on a cart.
The goal is to keep the pole upright by pushing the cart left or right.

**State space (4 continuous values):**

| Index | Variable | Typical range |
|-------|----------|---------------|
| 0 | Cart position | [-4.8, 4.8] |
| 1 | Cart velocity | (-inf, inf) |
| 2 | Pole angle (rad) | [-0.418, 0.418] |
| 3 | Pole angular velocity | (-inf, inf) |

**Actions:** 0 = push left, 1 = push right  
**Reward:** +1 for every step the pole stays up  
**Episode ends:** pole angle > 12 degrees, cart out of bounds, or 500 steps reached

A score of 500 means the agent balanced perfectly for the full episode.

In [ ]:
# Purpose: Inspect CartPole's observation and action spaces
# Key Concept: Always inspect env.observation_space and env.action_space before building an agent

env = gym.make('CartPole-v1')
obs, info = env.reset(seed=42)

print("Observation space:", env.observation_space)
print("Action space:     ", env.action_space)
print("Sample observation:", obs)
print()
print("State components:")
labels = ['cart position', 'cart velocity', 'pole angle (rad)', 'pole angular velocity']
for label, val in zip(labels, obs):
    print(f"  {label:30s}: {val:+.4f}")

## Discretizing the State Space

A Q-table needs discrete states. We bin each of the 4 continuous variables into
`n_bins` buckets. With 10 bins per dimension, that is $10^4 = 10{,}000$ possible
states — manageable, though many will be rarely visited.

**Trade-off:** More bins = finer resolution but a larger, slower-to-fill Q-table.
This trade-off disappears when we use neural networks (Quick-Start 02-03).

We clip velocities to practical ranges because their theoretical bounds are infinite.

In [ ]:
# Purpose: Build a discretizer that converts continuous observations to integer tuples
# Key Concept: Discretization is the bridge between continuous environments and Q-tables

class StateDiscretizer:
    """
    Bins each continuous observation dimension into one of `n_bins` buckets.

    Parameters
    ----------
    low : array-like
        Lower bound for each observation dimension.
    high : array-like
        Upper bound for each observation dimension.
    n_bins : int
        Number of bins per dimension.
    """

    def __init__(self, low, high, n_bins=10):
        self.low = np.array(low)
        self.high = np.array(high)
        self.n_bins = n_bins
        # Precompute bin edges for each dimension
        self.bins = [
            np.linspace(lo, hi, n_bins + 1)[1:-1]  # n_bins-1 interior edges
            for lo, hi in zip(self.low, self.high)
        ]

    def discretize(self, obs):
        """
        Convert a continuous observation array to a tuple of bin indices.

        Parameters
        ----------
        obs : np.ndarray
            Raw observation from the environment.

        Returns
        -------
        tuple of int
            Bin index for each dimension; usable as a Q-table key.
        """
        # Clip first so out-of-range values land in the boundary bin
        clipped = np.clip(obs, self.low, self.high)
        return tuple(int(np.digitize(clipped[i], self.bins[i])) for i in range(len(obs)))


# Practical bounds — velocities are theoretically unbounded, so we clip at ±3
OBS_LOW  = [-4.8, -3.0, -0.418, -3.0]
OBS_HIGH = [ 4.8,  3.0,  0.418,  3.0]
N_BINS   = 10

discretizer = StateDiscretizer(OBS_LOW, OBS_HIGH, n_bins=N_BINS)

# Verify: one raw observation -> one discrete state tuple
sample_obs = env.observation_space.sample()
discrete_state = discretizer.discretize(sample_obs)
print(f"Raw obs:      {sample_obs}")
print(f"Discrete key: {discrete_state}")
print(f"Total possible states: {N_BINS**4:,}")

## The Q-Learning Agent

Same Q-learning update rule as Quick-Start 00, now operating on discretized states.
Epsilon decays linearly from 1.0 to 0.05 over the first 80% of training so the agent
explores aggressively early and exploits toward the end.

In [ ]:
# Purpose: Define the Q-learning agent with linear epsilon decay
# Key Concept: Linear decay gives a predictable exploration schedule, unlike exponential decay

class CartPoleQLearner:
    """
    Tabular Q-learning agent with epsilon-greedy exploration.

    Parameters
    ----------
    n_actions : int
        Number of discrete actions (2 for CartPole).
    alpha : float
        Learning rate.
    gamma : float
        Discount factor.
    epsilon_start : float
        Initial exploration rate.
    epsilon_end : float
        Minimum exploration rate.
    n_episodes : int
        Total training episodes (used to schedule epsilon).
    """

    def __init__(self, n_actions=2, alpha=0.2, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.05, n_episodes=500):
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.n_episodes = n_episodes
        self.q_table = defaultdict(lambda: np.zeros(n_actions))
        self._episode = 0

    def select_action(self, state):
        """Epsilon-greedy selection."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        return int(np.argmax(self.q_table[state]))

    def update(self, state, action, reward, next_state, done):
        """Bellman update."""
        current_q = self.q_table[state][action]
        if done:
            target = reward
        else:
            target = reward + self.gamma * np.max(self.q_table[next_state])
        self.q_table[state][action] += self.alpha * (target - current_q)

    def end_episode(self):
        """Linear epsilon decay — called once per episode."""
        self._episode += 1
        frac = min(1.0, self._episode / (0.8 * self.n_episodes))
        self.epsilon = self.epsilon_start + frac * (self.epsilon_end - self.epsilon_start)


print("Agent class defined.")

## Training Loop — 500 Episodes

500 episodes trains in well under a minute. We record three things per episode:

- **Score** (= total reward = number of steps the pole stayed up)
- **Epsilon** (so we can see when exploration tapers off)
- **Unique states visited** (to check whether the Q-table is growing)

In [ ]:
# Purpose: Train the Q-learning agent and record diagnostics
# Key Concept: Logging during training is as important as the training itself

N_EPISODES = 500

np.random.seed(42)
env = gym.make('CartPole-v1')
agent = CartPoleQLearner(
    n_actions=2, alpha=0.2, gamma=0.99,
    epsilon_start=1.0, epsilon_end=0.05, n_episodes=N_EPISODES
)

scores   = []    # episode total reward (= steps survived)
epsilons = []    # exploration rate per episode

for episode in range(N_EPISODES):
    obs, _ = env.reset(seed=episode)   # different seed each episode
    state  = discretizer.discretize(obs)
    total_reward = 0
    done = False

    while not done:
        action = agent.select_action(state)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state = discretizer.discretize(next_obs)

        # Penalise termination to discourage falling quickly
        shaped_reward = reward if not terminated else -10.0
        agent.update(state, action, shaped_reward, next_state, done)

        state = next_state
        total_reward += reward   # log unmodified reward

    agent.end_episode()
    scores.append(total_reward)
    epsilons.append(agent.epsilon)

    if (episode + 1) % 100 == 0:
        recent_mean = np.mean(scores[-100:])
        print(f"Episode {episode+1:4d} | mean(last 100): {recent_mean:6.1f} "
              f"| epsilon: {agent.epsilon:.3f} "
              f"| Q-table states: {len(agent.q_table):,}")

env.close()

## Results: Reward Curve and Exploration Schedule

Two panels:
- **Left:** Episode score with a 20-episode rolling average
- **Right:** Epsilon schedule showing when exploration drops

A successful training run shows scores rising from near-zero (random stumbling) toward
200+ once the agent learns a consistent balancing strategy.

In [ ]:
# Purpose: Visualize training progress
# Key Concept: Plot epsilon alongside rewards to understand when exploitation takes over

def rolling_mean(data, window=20):
    return np.convolve(data, np.ones(window) / window, mode='valid')


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: episode scores ---
ax = axes[0]
episodes = np.arange(1, N_EPISODES + 1)
ax.plot(episodes, scores, color='#3498db', alpha=0.3, linewidth=0.8, label='Per-episode score')
roll = rolling_mean(scores, window=20)
ax.plot(np.arange(20, N_EPISODES + 1), roll, color='#1a5276', linewidth=2.5,
        label='Rolling mean (20 ep)')
ax.axhline(200, color='#e74c3c', linestyle='--', linewidth=1.2, label='Score = 200 (good)')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Score (steps survived)', fontsize=12)
ax.set_title('Q-Learning on CartPole-v1', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Right: epsilon over time ---
ax = axes[1]
ax.plot(episodes, epsilons, color='#8e44ad', linewidth=2)
ax.fill_between(episodes, epsilons, alpha=0.2, color='#8e44ad')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Epsilon (exploration rate)', fontsize=12)
ax.set_title('Exploration Schedule', fontsize=13)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

plt.suptitle('Tabular Q-Learning Training on CartPole-v1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nFinal 50-episode mean score: {np.mean(scores[-50:]):.1f}")
print(f"Best episode score:          {max(scores)}")
print(f"Q-table size:                {len(agent.q_table):,} unique states visited")

## Policy Snapshot: What Does the Agent Prefer?

We evaluate the fully-greedy policy (epsilon = 0) over 10 test episodes to measure
actual performance after training. During training, exploration noise masks true ability.

In [ ]:
# Purpose: Evaluate the trained policy without any exploration noise
# Key Concept: Always separate training performance (noisy) from evaluation performance (clean)

eval_env = gym.make('CartPole-v1')
eval_scores = []

for test_ep in range(10):
    obs, _ = eval_env.reset(seed=1000 + test_ep)
    state  = discretizer.discretize(obs)
    total  = 0
    done   = False
    while not done:
        # Greedy: always pick the highest Q-value action
        action = int(np.argmax(agent.q_table[state]))
        obs, reward, terminated, truncated, _ = eval_env.step(action)
        done  = terminated or truncated
        state = discretizer.discretize(obs)
        total += reward
    eval_scores.append(total)

eval_env.close()

print("Greedy evaluation over 10 episodes:")
for i, s in enumerate(eval_scores):
    print(f"  Test episode {i+1:2d}: score = {int(s):3d}")
print(f"\nMean eval score: {np.mean(eval_scores):.1f} / 500")

## Where Tabular Q-Learning Breaks Down

CartPole has only 4 state dimensions. Even so, our Q-table visited
thousands of unique states and many were visited rarely — leading to noisy Q-values.

Problems with tabular methods at scale:

1. **Curse of dimensionality** — 20 dimensions with 10 bins each = $10^{20}$ states
2. **No generalisation** — the agent learns nothing about state `(5, 3, 2, 1)` from
   visiting the similar state `(5, 3, 2, 2)`
3. **Memory** — large tables do not fit in RAM

The solution is to replace the table with a neural network that can generalise across
similar states. That is exactly what Quick-Start 02 (REINFORCE) and 03 (DQN) show.

## Key Takeaways

1. **Continuous state spaces** require discretization for tabular methods — bin count
   is a hyperparameter you tune
2. **Reward shaping** (the -10 terminal penalty) guides learning by making failure
   more salient than the default -0 signal
3. **Training score** includes exploration noise; always report a separate clean
   evaluation
4. **Q-table coverage** — the number of unique states visited — tells you whether
   the agent is exploring the state space adequately

## What's Next

Tabular Q-learning tops out around 200-300 on CartPole because discretization loses
information. In **Quick-Start 02**, a small neural network replaces the Q-table and
learns a continuous policy function — no binning required. For a full treatment of
Q-learning theory, see `modules/module_02_q_learning/` in this course.